<left>
<table style="margin-top:0px; margin-left:0px;">
<tr>
  <td><img src="https://raw.githubusercontent.com/worm-portal/WORM-Figures/master/style/worm.png" alt="WORM" title="WORM" width=75/></td>
  <td><h1 style=font-size:30px>Microbial energy supplies in a marine hydrothermal vent mixing zone</h1><br />
</tr>
</table>
</left>

**Authors: Grayson Boyer and Nuri Park**

*School of Earth and Space Exploration, Arizona State University*

Seawater percolates into the seafloor where it gets heated up and reacts with rocks to form hydrothermal fluid. This reduced fluid mixes back into oxidized seawater at marine vent systems. These systems are populated by microbes that take advantage of the mixed fluids by catalyzing catalyzing oxidation-reduction reactions to yield metabolic energy.

<img src="https://raw.githubusercontent.com/worm-portal/WORM-Figures/refs/heads/master/5-Community-WORM-Notebooks/Vent-Fluid-Mixing-2026/schematic.png" alt="Marine hydrothermal vent fluid mixing path" title="Marine hydrothermal vent fluid mixing path" width=50%/>

**Figure 1.** Fluid path of seawater becoming hydrothermal fluid and then mixing back into seawater in a vent mixing zone.

In this tutorial, we will model each step of this fluid mixing path:
1) speciate 2 °C seawater
2) react seawater with rock and heat to form 300 °C vent fluid. Basalt, basaltic andesite, and ultramafic rock compositions will be explored to produce different vent fluids that we can compare.
3) hot vent fluids will be mixed back into our original cold seawater so we can predict which reactions yield metabolic energy for microbes in the mixing zone. Does mixing from vent fluids created from different rock compositions favor different metabolic reactions?

In [1]:
# boilerplate cell that will be removed upon release
import sys
from pathlib import Path
import os
import pychnosz

# Add parent directory to Python path
parent_dir = Path.cwd().parent
sys.path.insert(0, str(parent_dir))

Load necessary python packages:

In [2]:
import aqequil
import pandas as pd

### Preparing Thermodynamic Databases for Water-Rock Reactions and Mixing

We will need two streamlined thermodynamic databases for:

1. heating up seawater and equilibrating with rock to form vent fluid. Since longer timescales are assumed, oxidized solutes in seawater will be allowed to become reduced.

2. mixing vent fluid with seawater. Shorter timescales are assumed so abiotic oxidation-reduction reactions are assumed to be kinetically inhibited. We will suppress abiotic redox reactions for iron, carbon, and sulfur species in the mixing zone. By doing this, we can calculate how much metabolic energy is available for reactions involving iron, carbon, and sulfur due to redox disequilibrium.

Streamlined databases are referred to by 3 letter codes, so let's create "wrr" (water-rock reaction) and "mix" (for mixing).

In [3]:
# Let's assume 250 bars pressure for our seafloor system
pressure = 250 # bars

# The water-rock reaction (wrr) database will allow all redox reactions
ae=aqequil.AqEquil(exclude_organics=True)
ae.create_data0(db="wrr", grid_press=[pressure]*8)

# The fluid mixing reaction (mix) will suppress redox reactions for species containing Fe, C, S
ae=aqequil.AqEquil(exclude_organics=True, suppress_redox=["Fe", "C", "S"])
ae.create_data0(db="mix", grid_press=[pressure]*8)

Loading Water-Organic-Rock-Microbe (WORM) thermodynamic databases...
Excluding ['organic_aq', 'organic_cr'] from column 'category_1'in wrm_data_latest.csv
wrm_data_latest.csv is now set as the active thermodynamic database.
Element database elements.csv is active.
Solid solution database solid_solutions.csv is active.
LogK database wrm_data_logK.csv is active.
Excluding ['organic_aq', 'organic_cr'] from column 'category_1'in wrm_data_logK.csv
LogK_S database wrm_data_logK_S.csv is active.
Excluding ['organic_aq', 'organic_cr'] from column 'category_1'in wrm_data_logK_S.csv
Loading thermodynamic database into pyCHNOSZ...
Creating data0.wrr...
Finished creating data0.wrr.
Loading Water-Organic-Rock-Microbe (WORM) thermodynamic databases...
Excluding ['organic_aq', 'organic_cr'] from column 'category_1'in wrm_data_latest.csv
wrm_data_latest.csv is now set as the active thermodynamic database.
Element database elements.csv is active.
Solid solution database solid_solutions.csv is active.
L

### Generating Vent Fluid from Seawater

Load the water-rock reaction database we just created:

In [4]:
ae=aqequil.AqEquil(db="wrr")

Loading a user-supplied thermodynamic database...
data1.wrr was not found in the EQ36DA directory but a data0.wrr was found in the current working directory. Using it...
data0.wrr is now set as the active thermodynamic database.


Take a quick look at the `seawater.csv` file included with this tutorial. It contains a single sample called BSW (bottom sea water or basal sea water). It is cold (2 °C), circumneutral (pH 7.8), and oxidized ($\log f_{O_2}$ = -0.9831).

We need to speciate our seawater before we can react it with rock. If there is any charge imbalance, Cl- will be adjusted to reach charge balance:

In [5]:
# speciating seawater, redox reactions allowed
speciation_seawater = ae.speciate(input_filename="seawater.csv",
                                  charge_balance_on="Cl-")

The input file column 'logfO2' will be used to set sample redox state. If a another column is desired, set it manually using the redox_flag parameter.
Successfully created a data1.wrr from data0.wrr
Using wrr to speciate BSW
Finished!


Now take a look at the `rock_compositions.csv` included with this tutorial. It contains three different rock compositions taken from the [GEOROC database](https://georoc.eu).
1. basalt
2. basaltic andesite
3. ultramafic rock

Values are in moles and are normalized to 1kg of major rock forming elements in each sample.

Let's react our seawater with these rock compositions and heat to 300 °C to model hydrothermal fluid generation:

In [6]:
# creates a table for mixing calculations
df = aqequil.react_water_rock(
        speciation = speciation_seawater,
        rock_composition = "rock_compositions.csv",
        end_T_C = 300, # °C
        db_next = "mix", # output will be formatted to use with the 'mix' database
        aux_basis = ["NH4+", "H2"],
        minimum_molality = 1E-18
        )

Using data0.wrr to react BSW
Using data0.wrr to react BSW
Using data0.wrr to react BSW


This produces a table that looks very similar to `seawater.csv` but includes rows for vent fluids that have equilibrated with the three rock compositions at 300 °C:

In [7]:
df

,Sample,H+,Pressure,Temperature,logfO2,Na+,Mg+2,Ca+2,K+,Cl-,...,Fe+2,Fe+3,CH4,NH4+,CO,H2,S2(g),S2-2,S2O3-2,SO3-2
0,,pH,bar,degC,logfO2,Molality,Molality,Molality,Molality,Molality,...,Molality,Molality,Molality,Molality,Molality,Molality,Molality,Molality,Molality,Molality
1,BSW,7.8,250.0,2,-0.9831,4.64E-01,5.27E-02,1.02E-02,2.30E-03,5.46E-01,...,3.00E-11,1.50E-09,3.00E-10,1.00E-12,1E-18,1E-18,1E-18,1E-18,1E-18,1E-18
2,basalt,6.3504,250.0,300.0,-37.244,5.3998E-01,8.4039E-07,2.1245E-03,5.2607E-03,5.4803E-01,...,1.7740E-07,7.8658E-10,2.3607E-03,3.0792E-07,4.9907E-11,2.0792E-02,1.4263E-16,1.8813E-11,1.0000E-18,6.7199E-16
3,basaltic andesite,5.801,250.0,300.0,-32.522,5.0703E-01,5.5891E-06,1.0496E-02,1.6422E-02,5.4419E-01,...,5.5026E-07,1.8475E-09,5.6520E-08,2.4306E-07,1.4440E-08,9.0719E-05,6.0844E-14,7.4812E-11,1.0389E-11,1.3049E-09
4,ultramafic rock,6.2896,250.0,300.0,-39.333,5.6548E-01,3.8082E-04,2.7639E-02,7.7705E-03,6.2864E-01,...,8.4728E-07,7.4790E-10,2.7079E-03,3.5321E-07,4.2547E-14,2.2541E-01,1.0000E-18,1.1333E-13,1.0000E-18,1.0000E-18


Concentrations haven't changed for our original BSW seawater sample. However, there are new columns representing concentrations of solutes that were not present in our original seawater. For example, there is now a column for carbon monoxide, CO, and sulfite, SO3-2 (among others). Since abiotic redox reactions are allowed, some of the original carbon and sulfur solutes present in our seawater reacted to form other carbon and sulfur solutes with different oxidation states.

Let's save this table as a CSV file:

In [8]:
df.to_csv("joined_wrr.csv", index=False)

You can open `joined_wrr.csv` and take a closer look if you'd like. Many concentrations are given as 1E-18 molal. This value was set earlier during the water-rock reaction calculation step as our minimum molality for reporting concentrations. A value of 1E-18 molal is very small (too dilute for microbes to "feel" and below the detection limit of most instruments) but is large enough to allow the speciation code to solve the system.

### Mixing Vent Fluid with Seawater

Now that we have seawater and three vent fluids together in a single CSV file, we can perform a mixing calculation. Let's load our thermodynamic database for mixing ("mix") that disallows abiotic redox reactions for iron, carbon, and sulfur species:

In [9]:
import aqequil # we can (re)start from this cell if we close the notebook
import pandas as pd

ae=aqequil.AqEquil(db="mix")

Loading a user-supplied thermodynamic database...
data1.mix was not found in the EQ36DA directory but a data0.mix was found in the current working directory. Using it...
data0.mix is now set as the active thermodynamic database.


Let's use this mixing database to speciate our seawater and vent fluids:

In [10]:
# speciating seawater and vent fluid, redox reactions for Fe, C, S not allowed
speciation_for_mix = ae.speciate(input_filename="joined_wrr.csv",
                                 charge_balance_on="Cl-")

The input file column 'logfO2' will be used to set sample redox state. If a another column is desired, set it manually using the redox_flag parameter.
Successfully created a data1.mix from data0.mix
Using mix to speciate BSW
Using mix to speciate basalt
Using mix to speciate basaltic andesite
Using mix to speciate ultramafic rock
Finished!


We've been using streamlined thermodynamic databases with three letter codes up until this point. These databases allow fast equilibration calculations but don't contain information that we need to perform microbial energy supply, so let's attach the main WORM thermodynamic database CSV file to allow those calculations later:

In [11]:
speciation_for_mix.thermo.csv_db = pd.read_csv("wrm_data_latest.csv")

Now let's mix our seawater with vent fluid. To start, let's mix samples "BSW" with "basalt"-reacted vent fluid.

In [12]:
mix_result_bst = aqequil.mix(speciation=speciation_for_mix, fluid_1="BSW", fluid_2="basalt")
mix_result_and = aqequil.mix(speciation=speciation_for_mix, fluid_1="BSW", fluid_2="basaltic andesite")
mix_result_ulm = aqequil.mix(speciation=speciation_for_mix, fluid_1="BSW", fluid_2="ultramafic rock")

Using data0.mix to react BSW
Using data0.mix to react basalt
Using data0.mix to react BSW
Using data0.mix to react basaltic.andesite
Using data0.mix to react BSW
Using data0.mix to react ultramafic.rock


How does pH change along our mixing gradient from pure cold seawater to pure hot vent fluid?

In [13]:
fig_bst = mix_result_bst.plot_pH(x_type="temperature")
fig_and = mix_result_and.plot_pH(x_type="temperature")
fig_ulm = mix_result_ulm.plot_pH(x_type="temperature")

aqequil.combine_plots(plot_list=[fig_bst, fig_and, fig_ulm],
              plot_titles=["Basalt", "B. Andesite", "Ultramafic"])

How do concentrations of iron, carbon, and sulfur oxidation states comprising dissolved species change across the mixing gradient?

In [14]:
plot_params = {
    "x_type":"temperature",
    "plot_elements":["Fe+2", "Fe+3", "C-4", "C+4", "S-2", "S+6"],
    "charge_sign_at_end":True,
    "log":True
}

fig_bst = mix_result_bst.plot_elements(**plot_params)
fig_and = mix_result_and.plot_elements(**plot_params)
fig_ulm = mix_result_ulm.plot_elements(**plot_params)

aqequil.combine_plots(plot_list=[fig_bst, fig_and, fig_ulm],
              plot_titles=["Basalt", "B. Andesite", "Ultramafic"])

### Predicted Microbial Metabolic Energy Landscapes

What is the predicted Gibbs free energy yield of microbe-catalyzed ferrous iron oxidation with oxygen to ferric iron?

In [15]:
plot_params = {
        "species":["Fe+2", "O2", "H+", "Fe+3", "H2O"],
        "stoich":[-1, -0.25, -1, 1, 0.5],
        "x_type":"temperature",
        "y_type":"G",
        "y_units":"kJ",
        "charge_sign_at_end":True,
        "show_zero_line":True,
        }

fig_bst = mix_result_bst.plot_energy(**plot_params)
fig_and = mix_result_and.plot_energy(**plot_params)
fig_ulm = mix_result_ulm.plot_energy(**plot_params)


# add a dotted red vertical line at 122 °C representing
# the upper known temperature limit of life
for fig in [fig_bst, fig_and, fig_ulm]:
    fig.add_shape(
        type="line",
        x0=122, x1=122,
        y0=0, y1=1,
        yref="y domain",
        xref="x",
        line=dict(color="red", dash="dot", width=2)
    )


aqequil.combine_plots(
        plot_list=[fig_bst, fig_and, fig_ulm],
        plot_titles=["Basalt", "B. Andesite", "Ultramafic"],
        title=fig_bst.layout["title"]["text"],
        plot_margins=dict(l=70, r=40, t=80, b=60))

The vertical red dotted line represents the upper known temperature limit of life at 122 °C [(Clarke 2014)](https://ui.adsabs.harvard.edu/link_gateway/2014IJAsB..13..141C/doi:10.1017/S1473550413000438). Microbes are not known to live above this temperature in marine vents.

There appears to be plenty of metabolic energy available from iron oxidation in basaltic andesite-hosted hydrothermal systems (e.g., -69 kJ/mol of catalyzed reaction at 101 °C).

For sulfide oxidation with oxygen to sulfate normalized to moles of sulfur species:

In [16]:
plot_params = {
        "species":["H2S", "O2", "SO4-2", "H+"],
        "stoich":[-1, -2, 1, 2],
        "x_type":"temperature",
        "y_type":"G",
        "y_units":"kJ",
        "charge_sign_at_end":True,
        "ylab":"ΔG, kJ/mole S",
        "show_zero_line":True,
        }


fig_bst = mix_result_bst.plot_energy(**plot_params)
fig_and = mix_result_and.plot_energy(**plot_params)
fig_ulm = mix_result_ulm.plot_energy(**plot_params)


# add a dotted red vertical line at 122 °C representing
# the upper known temperature limit of life
for fig in [fig_bst, fig_and, fig_ulm]:
    fig.add_shape(
        type="line",
        x0=122, x1=122,
        y0=0, y1=1,
        yref="y domain",
        xref="x",
        line=dict(color="red", dash="dot", width=2)
    )


aqequil.combine_plots(
        plot_list=[fig_bst, fig_and, fig_ulm],
        plot_titles=["Basalt", "B. Andesite", "Ultramafic"],
        title=fig_bst.layout["title"]["text"],
        plot_margins=dict(l=70, r=40, t=80, b=60),
        plotly_kwargs=None)

Even more metabolic energy is available from sulfide oxidation in basaltic andesite-hosted hydrothermal systems (e.g., -745 kJ/mol at 101 °C). We predict that more energy is available from sulfide oxidation than iron oxidation per mole of catalyzed reaction.

The overall shape (but not magnitude) of the energy landscapes of sulfide and iron oxidation look somewhat similar from a microbe's perspective. Microbial energy is available in the 2-122 °C range in the basaltic andesite-derived fluid mixing scenario for both of these reactions and not in the others. Why?

One commonality is that both reactions involve dissolved oxygen as the oxidant. Let's check the concentration of aqueous O$_2$. Let's also plot dissolved H$_2$ while we're at it:

In [17]:
plot_params = {
    "x_type":"temperature",
    "plot_species":["O2", "H2"],
    "charge_sign_at_end":True,
    "y_type":"log molality",
}

fig_bst = mix_result_bst.plot_aqueous_species(**plot_params)
fig_and = mix_result_and.plot_aqueous_species(**plot_params)
fig_ulm = mix_result_ulm.plot_aqueous_species(**plot_params)

# add a dotted red vertical line at 122 C representing
# the upper known temperature limit of life
for fig in [fig_bst, fig_and, fig_ulm]:
    fig.add_shape(
        type="line",
        x0=122, x1=122,
        y0=0, y1=1,
        yref="y domain",
        xref="x",
        line=dict(color="red", dash="dot", width=2)
    )

aqequil.combine_plots(plot_list=[fig_bst, fig_and, fig_ulm],
              plot_titles=["Basalt", "B. Andesite", "Ultramafic"])

The mixing situation is much more oxidized in the basaltic andesite derived-fluid whereas the basalt and ultramafic rock samples are hydrogen-rich. Let's try a metabolic reaction that consumes hydrogen to see if it is favorable when basalt or ultramafic rock-derived fluid mixes with seawater.

Predicted microbial energy yield for hydrogenotrophic methanogenesis:

In [18]:
plot_params = {
        "species":["CO2", "H2", "CH4", "H2O"],
        "stoich":[-1, -4, 1, 2],
        "x_type":"temperature",
        "y_type":"G",
        "y_units":"kJ",
        "charge_sign_at_end":True,
        "show_zero_line":True,
        }

fig_bst = mix_result_bst.plot_energy(**plot_params)
fig_and = mix_result_and.plot_energy(**plot_params)
fig_ulm = mix_result_ulm.plot_energy(**plot_params)


# add a dotted red vertical line at 122 C representing
# the upper known temperature limit of life
for fig in [fig_bst, fig_and, fig_ulm]:
    fig.add_shape(
        type="line",
        x0=122, x1=122,
        y0=0, y1=1,
        yref="y domain",
        xref="x",
        line=dict(color="red", dash="dot", width=2)
    )


aqequil.combine_plots(
        plot_list=[fig_bst, fig_and, fig_ulm],
        plot_titles=["Basalt", "B. Andesite", "Ultramafic"],
        title=fig_bst.layout["title"]["text"],
        plot_margins=dict(l=70, r=40, t=80, b=60))

Indeed, a reaction that consumes hydrogen is favored in the hydrogen-rich mixing scenarios!

### Summary

Aqueous geochemical modeling makes it is possible to predict the chemical composition of seawater-derived vent fluid formed from equilibration with different rock compositions. Redox disequilibrium occurs when hot and reduced vent fluid is mixed back into seawater. Microbes living in vent systems profit from this disequilibrium by catalyzing energy-yielding metabolic reactions. Certain metabolic reactions are favored over others depending on the availability of oxidants and reductants in the mixed fluids, which is in turn dictated by the underlying host rocks.

(End of tutorial)